# Kaggle: env + LoRA SFT + submission.zip

Single notebook for the Nemotron reasoning challenge:

1. Reinstall PyTorch / deps (matches NVIDIA utility script pattern).
2. Load Nemotron-3-Nano-30B, attach LoRA (rank ≤ 32).
3. Supervised fine-tune on `train.csv` with the same user suffix the **metric** notebook appends (`\\boxed{}` instruction).
4. Save adapter to `/kaggle/working` and zip **`adapter_config.json`** + **`adapter_model.safetensors`** only.

**After cell 1:** Kaggle often needs **Runtime → Restart session**, then run from cell 2 onward so imports pick up `/kaggle/working` wheels.

**Inputs:** add competition data (`train.csv`). Optionally attach the **ryanholbrook/nvidia-utility-script** notebook output / CUTLASS path used in the official demo.

In [ ]:
# --- Cell 1: environment (run once; restart kernel if imports fail) ---
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONPATH"] = f"/kaggle/working:{env.get('PYTHONPATH', '')}"

commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "uv pip install --target=/kaggle/working --system --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128",
    "uv pip install --target=/kaggle/working --system --no-build-isolation 'causal-conv1d>=1.4.0'",
    "uv pip install --target=/kaggle/working --system --no-build-isolation 'git+https://github.com/state-spaces/mamba.git'",
    "uv pip install --target=/kaggle/working --system 'trl>=0.15.0' 'datasets>=2.16.0' 'accelerate>=0.33.0' 'peft>=0.12.0' 'transformers>=4.44.0'",
]

for cmd in commands:
    print(f"Running: {cmd}")
    subprocess.run(cmd, shell=True, check=True, env=env)

print("Done. If the next cell fails to import torch, restart the kernel and skip this cell.")

In [ ]:
# --- Cell 2: paths + imports ---
import site
import sys

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
if __import__("pathlib").Path(cutlass_pkg_path).exists():
    site.addsitedir(cutlass_pkg_path)

import pandas as pd
import torch
import kagglehub
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    import mamba_ssm  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "mamba_ssm import failed. Run cell 1, restart kernel, then re-run from cell 2."
    ) from exc

OUTPUT_DIR = "/kaggle/working"
MODEL_ID = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
LORA_RANK = 32

# Training knobs (reduce for smoke tests)
MAX_TRAIN_SAMPLES = None  # e.g. 256 for a quick run
NUM_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 4096

# Match nvidia-nemotron-metric.ipynb user suffix
BOXED_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
# --- Cell 3: resolve train.csv (Kaggle competition input or cwd) ---
from pathlib import Path

def resolve_train_csv() -> Path:
    candidates = [
        Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),
        Path("/kaggle/input/competitions/nvidia-nemotron-3-reasoning-challenge/train.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    found = list(Path("/kaggle/input").rglob("train.csv")) if Path("/kaggle/input").exists() else []
    if found:
        return found[0]
    local = Path("train.csv")
    if local.exists():
        return local.resolve()
    raise FileNotFoundError("train.csv not found. Add the competition dataset as a Kaggle input.")

train_path = resolve_train_csv()
df = pd.read_csv(train_path)
if MAX_TRAIN_SAMPLES is not None:
    df = df.head(int(MAX_TRAIN_SAMPLES)).copy()
print("train_path:", train_path, "rows:", len(df))

In [ ]:
# --- Cell 4: tokenizer + chat formatting (mirrors metric + docs/training-data-format.md) ---
MODEL_PATH = kagglehub.model_download(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token


def example_to_text(prompt: str, answer: str) -> str:
    user_content = str(prompt).strip() + BOXED_SUFFIX
    assistant_content = f"\\boxed{{{str(answer).strip()}}}"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]
    kwargs = dict(tokenize=False, add_generation_prompt=False)
    try:
        return tokenizer.apply_chat_template(messages, **kwargs, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)


texts = [example_to_text(row.prompt, row.answer) for row in df.itertuples(index=False)]
train_ds = Dataset.from_dict({"text": texts})
print("sample text length (chars):", len(texts[0]) if texts else 0)

In [ ]:
# --- Cell 5: model + LoRA ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# --- Cell 6: SFT (TRL) ---
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=f"{OUTPUT_DIR}/sft_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

try:
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        tokenizer=tokenizer,
    )
trainer.train()

In [ ]:
# --- Cell 7: save adapter ---
model.save_pretrained(OUTPUT_DIR)
print("Saved adapter under", OUTPUT_DIR)

In [ ]:
# --- Cell 8: submission.zip (adapter files only) ---
import subprocess
from pathlib import Path

out = Path(OUTPUT_DIR)
cfg = out / "adapter_config.json"
weights = out / "adapter_model.safetensors"
if not weights.exists():
    weights = out / "adapter_model.bin"
if not cfg.exists() or not weights.exists():
    raise FileNotFoundError(f"Missing adapter files. Expected {cfg} and a weights file.")

subprocess.run(
    f"cd {OUTPUT_DIR} && zip -m submission.zip adapter_config.json {weights.name}",
    shell=True,
    check=True,
)
print("Wrote", out / "submission.zip")